In [13]:
from collections.abc import Iterable

import torch 

In [22]:
from torch.nn.utils.clip_grad import clip_grad_norm_

tensors = [torch.randn((5, 5)) for _ in range(6)]
max_norm = 1e-2

t1 = tuple(torch.nn.Parameter(torch.clone(t)) for t in tensors)
# Test freezing one parameter.
t1[-1].requires_grad_(False)

loss = torch.cat(t1).sum()
loss.backward()
t1_grads_prev = [torch.clone(t.grad) for t in t1 if t.grad is not None]
clip_grad_norm_(t1, max_norm)
t1_grads = [torch.clone(t.grad) for t in t1 if t.grad is not None]

In [24]:
t1_grads_prev[0]

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [54]:
norm = torch.linalg.vector_norm(t1_grads_prev[0], ord=2.0)
norm

tensor(5.)

In [64]:
coef = max_norm / (norm + 1e-6)
coef

tensor(0.0020)

In [62]:
max_norm / 0.0009

11.111111111111112

In [25]:
t1_grads[0]

tensor([[0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
        [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
        [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
        [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
        [0.0009, 0.0009, 0.0009, 0.0009, 0.0009]])

In [65]:
@torch.no_grad()
def gradient_clipping(
    parameters: Iterable[torch.nn.Parameter],
    max_l2_norm: float,
    eps: float = 1e-6
):
    gradients = [
        p.grad for p in parameters if not p.grad is None
    ]
    norm = torch.linalg.vector_norm(torch.stack(gradients), ord=2)
    for p in parameters:
        if p.grad is None:
            continue
        # norm = torch.linalg.vector_norm(p.grad, ord=2)
        coef = max_l2_norm / (norm + eps)
        coef = torch.clamp(coef, max=1.0)
        p.grad.mul_(coef)  # in-place scale
        # if coef < 1.0:
        #     p.grad.mul_(coef)  # in-place scale
    return

In [67]:
t1_c = tuple(torch.nn.Parameter(torch.clone(t)) for t in tensors)
t1_c[-1].requires_grad_(False)
loss_c = torch.cat(t1_c).sum()
loss_c.backward()

In [68]:
for p in t1_c:
    print(p.grad)
    norm = torch.norm(p.grad, p=2)
    print(norm)
    break

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])
tensor(5.)


In [69]:
gradient_clipping(t1_c, max_norm)
t1_c_grads = [torch.clone(t.grad) for t in t1_c if t.grad is not None]

In [70]:
t1_c_grads

[tensor([[0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009]]),
 tensor([[0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009]]),
 tensor([[0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009]]),
 tensor([[0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009, 0.0009, 0.0009, 0.0009],
         [0.0009, 0.0009,

In [61]:
len(t1_grads) == len(t1_c_grads)

True